In [1]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 56.6 MB/s eta 0:00:00


In [2]:
pip install transformers scikit-learn datasets statsmodels tqdm

In [3]:
pip install torch

In [5]:
!pip install -q gdown

folder_id = "Datasets_W2VvsBERT"
!gdown --folder https://drive.google.com/drive/folders/1oUVDeHucEAohgSP9oPgMn0WwF0K91WI9?usp=sharing{Datasets_W2VvsBERT}

Retrieving folder contents
Processing file 1MBmopc1dgWCgvkwCO_7nWoHYhQGuaTe4 combined.csv
Processing file 1t6TY2GzXzAXnjXpsuKJZXH2CWdlFKZK3 MEN_dataset_natural_form_full.txt
Processing file 1ZdAZVhHKjcRTC3kT8naz4oeLsWq8ZQpf ratings.txt
Processing file 1ETxmlbrSl5i7e05jdXb2RK5LStN55_T8 SimLex-999.txt
Processing file 1xyY2Mv4I_SriwX0MFcgYJw0AzAm6EyGA SimVerb-3500.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1MBmopc1dgWCgvkwCO_7nWoHYhQGuaTe4
To: /content/Datasets_W2VvsBERT/combined.csv
100% 7.43k/7.43k [00:00<00:00, 32.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1t6TY2GzXzAXnjXpsuKJZXH2CWdlFKZK3
To: /content/Datasets_W2VvsBERT/MEN_dataset_natural_form_full.txt
100% 69.9k/69.9k [00:00<00:00, 10.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZdAZVhHKjcRTC3kT8naz4oeLsWq8ZQpf
To: /content/Datasets_W2VvsBERT/ratings.txt
100% 2.37M/2.37M [00:00<00:00, 5

In [6]:
import os
import re
import time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, ttest_rel

try:
    from statsmodels.stats.contingency_tables import mcnemar
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

RNG = np.random.default_rng(42)

# ============================================================
# SHARED MODEL LOADING
# ============================================================

def load_word2vec_model(pretrained=True, corpus_sentences=None):
    from gensim.models import Word2Vec
    import gensim.downloader as api

    if pretrained:
        print("Loading pretrained word2vec model (this may take a while)...")
        model = api.load("word2vec-google-news-300")
        return model, "pretrained"
    else:
        assert corpus_sentences is not None
        print("Training word2vec on provided corpus...")
        model = Word2Vec(
            sentences=corpus_sentences, vector_size=300, window=5,
            min_count=5, sg=1, workers=4, epochs=10
        )
        return model.wv, "trained"


def get_word2vec_vector(word, w2v_model):
    word = word.lower()
    if word in w2v_model:
        return w2v_model[word]
    return None


def load_bert_model(model_name="bert-base-uncased"):
    from transformers import BertModel, BertTokenizer
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name, output_hidden_states=True)
    model.eval()
    return tokenizer, model


def cosine_sim(v1, v2):
    if v1 is None or v2 is None:
        return None
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10))


# ============================================================
# CONTEXT-AWARE BERT WORD EMBEDDINGS (carrier-sentence protocol)
# ============================================================

CARRIER_TEMPLATE = "This is a {word}."


def get_bert_word_vector_in_context(word, tokenizer, model, layer=-1,
                                     carrier=CARRIER_TEMPLATE):
    import torch
    sentence = carrier.format(word=word.lower())
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        outputs = model(**inputs)
    hidden_states = outputs.hidden_states[layer][0]  # [seq_len, hidden_dim]

    carrier_stopwords = {"this", "is", "a", "."}
    target_positions = [
        i for i, t in enumerate(tokens)
        if t not in ("[CLS]", "[SEP]") and t.replace("##", "") not in carrier_stopwords
    ]
    if not target_positions:
        target_positions = [i for i, t in enumerate(tokens) if t not in ("[CLS]", "[SEP]")]

    word_vector = hidden_states[target_positions].mean(dim=0).numpy()
    return word_vector


def get_bert_word_vector_all_layers(word, tokenizer, model, carrier=CARRIER_TEMPLATE):
    import torch
    sentence = carrier.format(word=word.lower())
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        outputs = model(**inputs)
    all_hidden = outputs.hidden_states

    carrier_stopwords = {"this", "is", "a", "."}
    target_positions = [
        i for i, t in enumerate(tokens)
        if t not in ("[CLS]", "[SEP]") and t.replace("##", "") not in carrier_stopwords
    ]
    if not target_positions:
        target_positions = [i for i, t in enumerate(tokens) if t not in ("[CLS]", "[SEP]")]

    layer_vectors = {}
    for layer_idx, hidden in enumerate(all_hidden):
        vec = hidden[0][target_positions].mean(dim=0).numpy()
        layer_vectors[layer_idx] = vec
    return layer_vectors


def get_bert_contextual_word_vector(word, sentence, tokenizer, model, layer=-1):
    import torch
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        outputs = model(**inputs)
    hidden_states = outputs.hidden_states[layer][0]

    word_lower = word.lower()
    target_positions = [i for i, t in enumerate(tokens)
                         if word_lower in t.lower().replace("##", "")]
    if not target_positions:
        target_positions = [i for i, t in enumerate(tokens) if t not in ("[CLS]", "[SEP]")]

    return hidden_states[target_positions].mean(dim=0).numpy()


# ============================================================
# PART 1: WORD SIMILARITY -- MULTI-BENCHMARK EVALUATION
# ============================================================
def load_wordsim353(path="/combined.csv"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={
        "Word 1": "Word1", "Word 2": "Word2", "Human (mean)": "Score"
    })
    return df[["Word1", "Word2", "Score"]]


def load_simlex999(path="/SimLex-999.txt"):
    if not os.path.exists(path):
        print(f"[SimLex-999] File not found at '{path}'. Download it from "
              f"https://fh295.github.io/simlex.html and place it here. Skipping.")
        return None
    df = pd.read_csv(path, sep="\t")
    df = df.rename(columns={"word1": "Word1", "word2": "Word2", "SimLex999": "Score"})
    return df[["Word1", "Word2", "Score", "POS"]] if "POS" in df.columns else df[["Word1", "Word2", "Score"]]


def load_simverb3500(path="/SimVerb-3500.txt"):
    if not os.path.exists(path):
        print(f"[SimVerb-3500] File not found at '{path}'. Download it from "
              f"https://fh295.github.io/simlex.html and place it here. Skipping.")
        return None
    df = pd.read_csv(path, sep="\t", header=None,
                      names=["Word1", "Word2", "POS", "Score", "Relation"])
    return df[["Word1", "Word2", "Score"]]


def load_men(path="MEN_dataset_natural_form_full"):
    if not os.path.exists(path):
        print(f"[MEN] File not found at '{path}'. Download it from "
              f"https://staff.fnwi.uva.nl/e.bruni/MEN and place it here. Skipping.")
        return None
    df = pd.read_csv(path, sep=" ", header=None, names=["Word1", "Word2", "Score"])
    df["Word1"] = df["Word1"].str.replace(r"-[a-z]$", "", regex=True)
    df["Word2"] = df["Word2"].str.replace(r"-[a-z]$", "", regex=True)
    df["Score"] = df["Score"].astype(float) / 5.0  # 0-50 -> 0-10
    return df[["Word1", "Word2", "Score"]]


def load_scws(path="/ratings.txt"):
    if not os.path.exists(path):
        print(f"[SCWS] File not found at '{path}'. Download it from "
              f"https://ai.stanford.edu/~ehhuang/SCWS.zip and place it here. Skipping.")
        return None
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 8:
                continue
            rows.append({
                "Word1": parts[1], "Word2": parts[3],
                "Sentence1": parts[5], "Sentence2": parts[6],
                "Score": float(parts[7]),
            })
    return pd.DataFrame(rows)


def bootstrap_ci_corr_diff(human, scores_a, scores_b, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    human = np.asarray(human)
    scores_a = np.asarray(scores_a)
    scores_b = np.asarray(scores_b)
    n = len(human)

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        corr_a, _ = spearmanr(human[idx], scores_a[idx])
        corr_b, _ = spearmanr(human[idx], scores_b[idx])
        diffs.append(corr_a - corr_b)

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    point_diff = spearmanr(human, scores_a)[0] - spearmanr(human, scores_b)[0]
    return point_diff, ci_low, ci_high, diffs


def evaluate_word_similarity(df, w2v_model, tokenizer, bert_model, use_context=True,
                              dataset_name="WordSim-353"):
    results = []
    w2v_scores, bert_scores, human_scores = [], [], []
    n_total = len(df)
    n_oov = 0

    for _, row in df.iterrows():
        w1, w2, human_score = row["Word1"], row["Word2"], row["Score"]

        v1_w2v = get_word2vec_vector(w1, w2v_model)
        v2_w2v = get_word2vec_vector(w2, w2v_model)
        if v1_w2v is None or v2_w2v is None:
            n_oov += 1
            continue
        sim_w2v = cosine_sim(v1_w2v, v2_w2v)

        if use_context:
            v1_bert = get_bert_word_vector_in_context(w1, tokenizer, bert_model)
            v2_bert = get_bert_word_vector_in_context(w2, tokenizer, bert_model)
        else:
            v1_bert = get_bert_word_vector_in_context(w1, tokenizer, bert_model, carrier="{word}")
            v2_bert = get_bert_word_vector_in_context(w2, tokenizer, bert_model, carrier="{word}")
        sim_bert = cosine_sim(v1_bert, v2_bert)

        w2v_scores.append(sim_w2v)
        bert_scores.append(sim_bert)
        human_scores.append(human_score)
        results.append({"word1": w1, "word2": w2, "human": human_score,
                         "word2vec_sim": sim_w2v, "bert_sim": sim_bert})

    oov_rate = n_oov / n_total if n_total else np.nan
    print(f"\n[{dataset_name}] word2vec OOV drop rate: {n_oov}/{n_total} pairs "
          f"({oov_rate:.2%}) excluded due to at least one out-of-vocabulary word.")

    corr_w2v, _ = spearmanr(human_scores, w2v_scores)
    corr_bert, _ = spearmanr(human_scores, bert_scores)
    print(f"[{dataset_name}] Spearman (word2vec vs human): {corr_w2v:.4f} (n={len(human_scores)})")
    print(f"[{dataset_name}] Spearman (BERT vs human): {corr_bert:.4f} (n={len(human_scores)})")

    point_diff, ci_low, ci_high, _ = bootstrap_ci_corr_diff(human_scores, bert_scores, w2v_scores)
    print(f"[{dataset_name}] Bootstrap 95% CI (BERT - word2vec): "
          f"{point_diff:.4f} [{ci_low:.4f}, {ci_high:.4f}]")
    if ci_low > 0:
        print(f"[{dataset_name}] -> BERT's advantage is statistically robust (CI excludes 0).")
    elif ci_high < 0:
        print(f"[{dataset_name}] -> word2vec's advantage is statistically robust (CI excludes 0).")
    else:
        print(f"[{dataset_name}] -> Difference is NOT statistically significant (CI includes 0).")

    stats = {
        "dataset": dataset_name,
        "corr_w2v": corr_w2v, "corr_bert": corr_bert,
        "oov_rate": oov_rate, "n_oov": n_oov, "n_total": n_total,
        "corr_diff": point_diff, "ci_low": ci_low, "ci_high": ci_high,
    }
    return pd.DataFrame(results), stats


def evaluate_scws(df, w2v_model, tokenizer, bert_model, dataset_name="SCWS"):
    if df is None:
        return None, None

    results = []
    w2v_scores, bert_scores, human_scores = [], [], []
    n_total = len(df)
    n_oov = 0

    for _, row in df.iterrows():
        w1, w2 = row["Word1"], row["Word2"]
        s1, s2 = row["Sentence1"], row["Sentence2"]
        human_score = row["Score"]

        v1_w2v = get_word2vec_vector(w1, w2v_model)
        v2_w2v = get_word2vec_vector(w2, w2v_model)
        if v1_w2v is None or v2_w2v is None:
            n_oov += 1
            continue
        sim_w2v = cosine_sim(v1_w2v, v2_w2v)
        v1_bert = get_bert_contextual_word_vector(w1, s1, tokenizer, bert_model)
        v2_bert = get_bert_contextual_word_vector(w2, s2, tokenizer, bert_model)
        sim_bert = cosine_sim(v1_bert, v2_bert)

        w2v_scores.append(sim_w2v)
        bert_scores.append(sim_bert)
        human_scores.append(human_score)
        results.append({"word1": w1, "word2": w2, "sentence1": s1, "sentence2": s2,
                         "human": human_score, "word2vec_sim": sim_w2v, "bert_sim": sim_bert})

    oov_rate = n_oov / n_total if n_total else np.nan
    corr_w2v, _ = spearmanr(human_scores, w2v_scores)
    corr_bert, _ = spearmanr(human_scores, bert_scores)
    print(f"\n[{dataset_name}] word2vec OOV drop rate: {n_oov}/{n_total} ({oov_rate:.2%})")
    print(f"[{dataset_name}] Spearman (word2vec vs human, context-blind): {corr_w2v:.4f} (n={len(human_scores)})")
    print(f"[{dataset_name}] Spearman (BERT vs human, context-aware): {corr_bert:.4f} (n={len(human_scores)})")

    point_diff, ci_low, ci_high, _ = bootstrap_ci_corr_diff(human_scores, bert_scores, w2v_scores)
    print(f"[{dataset_name}] Bootstrap 95% CI (BERT - word2vec): {point_diff:.4f} [{ci_low:.4f}, {ci_high:.4f}]")

    stats = {
        "dataset": dataset_name,
        "corr_w2v": corr_w2v, "corr_bert": corr_bert,
        "oov_rate": oov_rate, "n_oov": n_oov, "n_total": n_total,
        "corr_diff": point_diff, "ci_low": ci_low, "ci_high": ci_high,
    }
    return pd.DataFrame(results), stats


def run_multi_benchmark_word_similarity(
    w2v_model,
    tokenizer,
    bert_model,
    wordsim_path="/combined.csv",
    simlex_path="/SimLex-999.txt",
    simverb_path="/SimVerb-3500.txt",
    men_path="MEN_dataset_natural_form_full",
    scws_path="/ratings.txt"
):
    all_stats = []
    all_pair_dfs = {}

    df_wordsim = load_wordsim353(wordsim_path)
    pairs_ws, stats_ws = evaluate_word_similarity(df_wordsim, w2v_model, tokenizer, bert_model, dataset_name="WordSim-353")
    all_stats.append(stats_ws)
    all_pair_dfs["wordsim353"] = pairs_ws

    df_simlex = load_simlex999(simlex_path)
    if df_simlex is not None:
        pairs_sl, stats_sl = evaluate_word_similarity(df_simlex, w2v_model, tokenizer, bert_model, dataset_name="SimLex-999")
        all_stats.append(stats_sl)
        all_pair_dfs["simlex999"] = pairs_sl

    df_simverb = load_simverb3500(simverb_path)
    if df_simverb is not None:
        pairs_sv, stats_sv = evaluate_word_similarity(df_simverb, w2v_model, tokenizer, bert_model, dataset_name="SimVerb-3500")
        all_stats.append(stats_sv)
        all_pair_dfs["simverb3500"] = pairs_sv

    df_men = load_men(men_path)
    if df_men is not None:
        pairs_men, stats_men = evaluate_word_similarity(df_men, w2v_model, tokenizer, bert_model, dataset_name="MEN")
        all_stats.append(stats_men)
        all_pair_dfs["men"] = pairs_men

    df_scws = load_scws(scws_path)
    if df_scws is not None:
        pairs_scws, stats_scws = evaluate_scws(df_scws, w2v_model, tokenizer, bert_model, dataset_name="SCWS")
        all_stats.append(stats_scws)
        all_pair_dfs["scws"] = pairs_scws

    comparison_df = pd.DataFrame(all_stats)
    print("\n=== MULTI-BENCHMARK WORD SIMILARITY COMPARISON ===")
    print(comparison_df.to_string(index=False))

    return comparison_df, all_pair_dfs

# ============================================================
# QUANTIFIED POLYSEMY EXPERIMENT
# ============================================================

POLYSEMY_DATA = {
    "bank": {
        "finance": ["I deposited money in the bank.",
                    "The bank approved my loan application.",
                    "She works as a manager at the bank."],
        "river": ["We sat on the bank of the river.",
                  "The flood eroded the river bank overnight.",
                  "Fishermen lined the bank at dawn."],
    },
    "crane": {
        "bird": ["The crane waded through the shallow marsh.",
                 "A crane flew gracefully over the wetland.",
                 "Bird watchers photographed the crane at sunrise."],
        "machine": ["The crane lifted the steel beam onto the roof.",
                     "Workers operated the crane at the construction site.",
                     "The tower crane collapsed during the storm."],
    },
    "bat": {
        "animal": ["The bat flew out of the cave at dusk.",
                   "A colony of bats roosted in the attic.",
                   "The bat used echolocation to find insects."],
        "sports": ["He swung the bat and hit a home run.",
                   "She bought a new baseball bat.",
                   "The cricket bat was made of willow."],
    },
    "spring": {
        "season": ["Flowers bloom in the spring.",
                    "We planted the garden this spring.",
                    "Spring brings warmer weather."],
        "coil": ["The mattress spring was broken.",
                 "He compressed the metal spring.",
                 "The spring mechanism powered the toy."],
        "water_source": ["They drank water from the mountain spring.",
                          "The spring fed a small stream.",
                          "A natural spring bubbled up from the rocks."],
    },
    "mouse": {
        "animal": ["The mouse scurried across the kitchen floor.",
                    "A cat chased the mouse into its hole.",
                    "The mouse nibbled on a piece of cheese."],
        "device": ["He clicked the mouse to open the file.",
                   "The wireless mouse needs new batteries.",
                   "She moved the mouse cursor across the screen."],
    },
    "plant": {
        "vegetation": ["She watered the plant every morning.",
                        "The plant grew tall in the sunlight.",
                        "He potted a new plant on the balcony."],
        "factory": ["The car plant employs thousands of workers.",
                     "The factory plant produces steel parts.",
                     "The power plant supplies electricity to the city."],
    },
    "star": {
        "celestial": ["The star twinkled in the night sky.",
                       "Astronomers observed a distant star.",
                       "The star exploded in a supernova."],
        "celebrity": ["The movie star signed autographs for fans.",
                       "She became a star after her debut album.",
                       "The football star scored the winning goal."],
    },
    "pitch": {
        "sports": ["The players ran onto the pitch.",
                    "The football pitch was freshly mowed.",
                    "The stadium pitch was flooded after the storm."],
        "sound": ["The singer hit a high pitch.",
                   "The pitch of the violin was slightly off.",
                   "Sound waves determine the pitch of a tone."],
        "sales": ["He gave a sales pitch to investors.",
                   "The startup's pitch impressed the judges.",
                   "She rehearsed her pitch before the meeting."],
    },
    "novel": {
        "book": ["She wrote a novel about her childhood.",
                  "The novel won a literary prize.",
                  "He is reading a mystery novel."],
        "new": ["The scientists proposed a novel approach.",
                 "This is a novel solution to an old problem.",
                 "The novel technique improved efficiency."],
    },
    "light": {
        "illumination": ["She turned on the light in the room.",
                          "The light from the lamp was too bright.",
                          "Sunlight filled the room through the window."],
        "weight": ["The suitcase was surprisingly light.",
                    "He preferred light backpacks for hiking.",
                    "The fabric felt light and airy."],
    },
    "rock": {
        "stone": ["He climbed over a large rock.",
                   "The rock formation was millions of years old.",
                   "She collected rocks along the beach."],
        "music": ["They listened to rock music all night.",
                   "The rock band performed at the festival.",
                   "He plays guitar in a rock group."],
    },
    "book": {
        "reading": ["She read a book before bed.",
                     "The library has thousands of books.",
                     "He wrote his first book last year."],
        "reserve": ["I need to book a flight to Paris.",
                     "She booked a table at the restaurant.",
                     "They booked the hotel room in advance."],
    },
}


def run_polysemy_experiment(tokenizer, bert_model, w2v_model, polysemy_data=POLYSEMY_DATA):
    import torch
    rows = []
    for word, senses in polysemy_data.items():
        sense_names = list(senses.keys())
        bert_vectors = {}
        for sense, sentences in senses.items():
            vecs = []
            for sent in sentences:
                inputs = tokenizer(sent, return_tensors="pt")
                tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
                with torch.no_grad():
                    outputs = bert_model(**inputs)
                hidden = outputs.hidden_states[-1][0]
                target_idx = [i for i, t in enumerate(tokens)
                              if word.lower() in t.lower().replace("##", "")]
                if not target_idx:
                    continue
                vecs.append(hidden[target_idx[0]].numpy())
            bert_vectors[sense] = vecs

        within_sims = []
        for sense, vecs in bert_vectors.items():
            for i in range(len(vecs)):
                for j in range(i + 1, len(vecs)):
                    within_sims.append(cosine_sim(vecs[i], vecs[j]))

        across_sims = []
        for i, s1 in enumerate(sense_names):
            for s2 in sense_names[i + 1:]:
                for v1 in bert_vectors[s1]:
                    for v2 in bert_vectors[s2]:
                        across_sims.append(cosine_sim(v1, v2))

        bert_within_mean = np.mean(within_sims) if within_sims else np.nan
        bert_across_mean = np.mean(across_sims) if across_sims else np.nan
        bert_gap = bert_within_mean - bert_across_mean

        w2v_vec = get_word2vec_vector(word, w2v_model)
        w2v_within_mean = 1.0 if w2v_vec is not None else np.nan
        w2v_across_mean = 1.0 if w2v_vec is not None else np.nan
        w2v_gap = 0.0 if w2v_vec is not None else np.nan

        rows.append({
            "word": word, "n_senses": len(sense_names),
            "bert_within_sense_sim": bert_within_mean,
            "bert_across_sense_sim": bert_across_mean,
            "bert_gap": bert_gap,
            "word2vec_within_sim": w2v_within_mean,
            "word2vec_across_sim": w2v_across_mean,
            "word2vec_gap": w2v_gap,
        })

    df = pd.DataFrame(rows)
    print("\n=== POLYSEMY EXPERIMENT SUMMARY ===")
    print(df.to_string(index=False))
    print(f"\nMean BERT gap (within - across sense similarity): {df['bert_gap'].mean():.4f}")
    print("Mean word2vec gap: 0.0000 (by construction).")

    df_sorted = df.sort_values("bert_gap", ascending=False)
    print("\nWords where BERT separates senses BEST (largest gap):")
    print(df_sorted[["word", "bert_gap"]].head(5).to_string(index=False))
    print("\nWords where BERT separates senses WORST (smallest gap):")
    print(df_sorted[["word", "bert_gap"]].tail(5).to_string(index=False))
    return df


def interpret_polysemy_results(df):
    homonym_like = ["bank", "bat", "mouse", "star", "book"]
    metaphor_like = ["crane", "spring", "plant", "pitch", "light", "rock", "novel"]

    h_gap = df[df["word"].isin(homonym_like)]["bert_gap"].mean()
    m_gap = df[df["word"].isin(metaphor_like)]["bert_gap"].mean()

    print("\n=== INTERPRETATION TEMPLATE ===")
    print(f"Average BERT gap for homonym-like words: {h_gap:.4f}")
    print(f"Average BERT gap for metaphor/extension-like words: {m_gap:.4f}")
    if h_gap > m_gap:
        print("-> Consistent with the hypothesis that BERT separates unrelated "
              "(homonymous) senses more easily than related/metaphorical extensions.")
    else:
        print("-> Contrary to the naive hypothesis -- inspect individual words.")
    return {"homonym_gap": h_gap, "metaphor_gap": m_gap}


# ============================================================
# LAYER-WISE PROBING
# ============================================================

def layer_wise_word_similarity_probe(df, w2v_model, tokenizer, bert_model, n_layers=13):
    layer_scores = {l: [] for l in range(n_layers)}
    human_scores = []
    for _, row in df.iterrows():
        w1, w2, human_score = row["Word1"], row["Word2"], row["Score"]
        vecs1 = get_bert_word_vector_all_layers(w1, tokenizer, bert_model)
        vecs2 = get_bert_word_vector_all_layers(w2, tokenizer, bert_model)
        human_scores.append(human_score)
        for l in range(n_layers):
            layer_scores[l].append(cosine_sim(vecs1[l], vecs2[l]))

    layer_corrs = {l: spearmanr(human_scores, layer_scores[l])[0] for l in range(n_layers)}
    result_df = pd.DataFrame({"layer": list(layer_corrs.keys()),
                               "spearman_corr_word_similarity": list(layer_corrs.values())})
    best_layer = result_df.loc[result_df["spearman_corr_word_similarity"].idxmax()]
    print("\n=== LAYER-WISE PROBING: WORD SIMILARITY ===")
    print(result_df.to_string(index=False))
    print(f"\nBest layer for word similarity: layer {int(best_layer['layer'])} "
          f"(Spearman r = {best_layer['spearman_corr_word_similarity']:.4f})")
    return result_df


def layer_wise_classification_probe(train_texts, train_labels, test_texts, test_labels,
                                     tokenizer, bert_model, n_layers=13):
    import torch
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score

    def embed_sentence_all_layers(sentence):
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True,
                           max_length=128, padding=True)
        with torch.no_grad():
            outputs = bert_model(**inputs)
        return [outputs.hidden_states[l][0].mean(dim=0).numpy() for l in range(n_layers)]

    print("\nExtracting layer-wise embeddings for classification probe...")
    train_layer_embeds = [[] for _ in range(n_layers)]
    for text in train_texts:
        vecs = embed_sentence_all_layers(text)
        for l in range(n_layers):
            train_layer_embeds[l].append(vecs[l])

    test_layer_embeds = [[] for _ in range(n_layers)]
    for text in test_texts:
        vecs = embed_sentence_all_layers(text)
        for l in range(n_layers):
            test_layer_embeds[l].append(vecs[l])

    rows = []
    for l in range(n_layers):
        X_train = np.array(train_layer_embeds[l])
        X_test = np.array(test_layer_embeds[l])
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, train_labels)
        preds = clf.predict(X_test)
        acc = accuracy_score(test_labels, preds)
        f1 = f1_score(test_labels, preds, average="weighted")
        rows.append({"layer": l, "accuracy": acc, "f1": f1})

    result_df = pd.DataFrame(rows)
    best_layer = result_df.loc[result_df["f1"].idxmax()]
    print("\n=== LAYER-WISE PROBING: TEXT CLASSIFICATION ===")
    print(result_df.to_string(index=False))
    print(f"\nBest layer for classification: layer {int(best_layer['layer'])} "
          f"(F1 = {best_layer['f1']:.4f})")
    return result_df


# ============================================================
# PART 2: TEXT CLASSIFICATION -- MULTI-SEED SST-2 EVALUATION
# ============================================================
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def get_sentence_embedding_w2v(sentence, w2v_model, dim=300):
    words = simple_tokenize(sentence)
    vectors = [w2v_model[w] for w in words if w in w2v_model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(dim)


def get_sentence_embedding_bert(sentence, tokenizer, bert_model, pooling="mean"):
    import torch
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True,
                       max_length=128, padding=True)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    if pooling == "cls":
        return outputs.last_hidden_state[0, 0].numpy()
    return outputs.last_hidden_state[0].mean(dim=0).numpy()


def load_sst2_stratified_subsample(n_train_per_class=250, n_test_per_class=100, seed=42):
    from datasets import load_dataset, concatenate_datasets

    print(f"Loading SST-2 stratified subsample (seed={seed})...")
    dataset = load_dataset("stanfordnlp/sst2")
    train, val = dataset["train"], dataset["validation"]

    train_pos = train.filter(lambda x: x["label"] == 1).shuffle(seed=seed).select(range(n_train_per_class))
    train_neg = train.filter(lambda x: x["label"] == 0).shuffle(seed=seed).select(range(n_train_per_class))
    train_subset = concatenate_datasets([train_pos, train_neg]).shuffle(seed=seed)

    val_pos = val.filter(lambda x: x["label"] == 1).shuffle(seed=seed).select(range(n_test_per_class))
    val_neg = val.filter(lambda x: x["label"] == 0).shuffle(seed=seed).select(range(n_test_per_class))
    test_subset = concatenate_datasets([val_pos, val_neg]).shuffle(seed=seed)

    print(f"[seed={seed}] Train size: {len(train_subset)}, Test size: {len(test_subset)}")
    return (train_subset["sentence"], train_subset["label"],
            test_subset["sentence"], test_subset["label"])


def run_classification_experiment(train_texts, train_labels, test_texts, test_labels,
                                   w2v_model, tokenizer, bert_model, verbose=True):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

    if verbose:
        print("Extracting word2vec embeddings (separately timed)...")
    t0 = time.time()
    X_train_w2v = np.array([get_sentence_embedding_w2v(t, w2v_model) for t in train_texts])
    X_test_w2v = np.array([get_sentence_embedding_w2v(t, w2v_model) for t in test_texts])
    t_w2v_extract = time.time() - t0

    if verbose:
        print("Extracting BERT embeddings (separately timed)...")
    t0 = time.time()
    X_train_bert = np.array([get_sentence_embedding_bert(t, tokenizer, bert_model) for t in train_texts])
    X_test_bert = np.array([get_sentence_embedding_bert(t, tokenizer, bert_model) for t in test_texts])
    t_bert_extract = time.time() - t0

    results = {}
    for name, X_train, X_test, extract_time in [
        ("word2vec", X_train_w2v, X_test_w2v, t_w2v_extract),
        ("bert", X_train_bert, X_test_bert, t_bert_extract),
    ]:
        clf = LogisticRegression(max_iter=1000)
        t0 = time.time()
        clf.fit(X_train, train_labels)
        train_time = time.time() - t0

        preds = clf.predict(X_test)
        acc = accuracy_score(test_labels, preds)
        f1 = f1_score(test_labels, preds, average="weighted")
        cm = confusion_matrix(test_labels, preds)

        results[name] = {
            "accuracy": acc, "f1": f1, "confusion_matrix": cm,
            "embedding_extraction_time": extract_time,
            "classifier_train_time": train_time,
            "predictions": preds,
        }
        if verbose:
            print(f"\n=== {name.upper()} RESULTS ===")
            print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
            print(classification_report(test_labels, preds))

    speedup = results["bert"]["embedding_extraction_time"] / max(results["word2vec"]["embedding_extraction_time"], 1e-9)
    if verbose:
        print(f"\nBERT embedding extraction took {speedup:.1f}x longer than word2vec.")
    return results


def error_analysis(test_texts, test_labels, results):
    preds_w2v = results["word2vec"]["predictions"]
    preds_bert = results["bert"]["predictions"]
    disagreements = [
        {"text": t, "true_label": y, "word2vec_pred": pw, "bert_pred": pb}
        for t, y, pw, pb in zip(test_texts, test_labels, preds_w2v, preds_bert)
        if pw != pb
    ]
    return pd.DataFrame(disagreements)


def run_multi_seed_classification(w2v_model, tokenizer, bert_model,
                                   seeds=(10, 42, 123, 999, 2024),
                                   n_train_per_class=250, n_test_per_class=100):
    per_seed_rows = []
    last_seed_results = None
    last_seed_test_labels = None

    for seed in seeds:
        train_texts, train_labels, test_texts, test_labels = load_sst2_stratified_subsample(
            n_train_per_class=n_train_per_class, n_test_per_class=n_test_per_class, seed=seed
        )
        results = run_classification_experiment(
            train_texts, train_labels, test_texts, test_labels,
            w2v_model, tokenizer, bert_model, verbose=False
        )
        per_seed_rows.append({
            "seed": seed,
            "w2v_accuracy": results["word2vec"]["accuracy"],
            "w2v_f1": results["word2vec"]["f1"],
            "bert_accuracy": results["bert"]["accuracy"],
            "bert_f1": results["bert"]["f1"],
        })
        print(f"[seed={seed}] w2v acc={results['word2vec']['accuracy']:.4f} f1={results['word2vec']['f1']:.4f} "
              f"| bert acc={results['bert']['accuracy']:.4f} f1={results['bert']['f1']:.4f}")

        last_seed_results = results
        last_seed_test_labels = test_labels

    per_seed_df = pd.DataFrame(per_seed_rows)

    summary = {
        "w2v_acc_mean": per_seed_df["w2v_accuracy"].mean(),
        "w2v_acc_std": per_seed_df["w2v_accuracy"].std(ddof=1),
        "bert_acc_mean": per_seed_df["bert_accuracy"].mean(),
        "bert_acc_std": per_seed_df["bert_accuracy"].std(ddof=1),
        "w2v_f1_mean": per_seed_df["w2v_f1"].mean(),
        "w2v_f1_std": per_seed_df["w2v_f1"].std(ddof=1),
        "bert_f1_mean": per_seed_df["bert_f1"].mean(),
        "bert_f1_std": per_seed_df["bert_f1"].std(ddof=1),
    }

    print("\n=== MULTI-SEED SST-2 CLASSIFICATION SUMMARY ===")
    print(per_seed_df.to_string(index=False))
    print(f"\nAccuracy: word2vec = {summary['w2v_acc_mean']:.4f} +/- {summary['w2v_acc_std']:.4f}, "
          f"BERT = {summary['bert_acc_mean']:.4f} +/- {summary['bert_acc_std']:.4f}")
    print(f"F1:       word2vec = {summary['w2v_f1_mean']:.4f} +/- {summary['w2v_f1_std']:.4f}, "
          f"BERT = {summary['bert_f1_mean']:.4f} +/- {summary['bert_f1_std']:.4f}")

    # Paired t-test across seeds (F1 as the paired statistic)
    t_stat, p_value = ttest_rel(per_seed_df["bert_f1"], per_seed_df["w2v_f1"])
    print(f"\nPaired t-test across {len(seeds)} seeds (BERT F1 vs word2vec F1): "
          f"t={t_stat:.4f}, p={p_value:.4f}")
    if p_value < 0.05:
        print("-> BERT's advantage is statistically significant across seeds (p < 0.05); "
              "the 4% single-split gap is NOT attributable to sampling noise from seed 42.")
    else:
        print("-> Not statistically significant at the 95% level across seeds -- treat the "
              "single-split gap with caution.")

    # McNemar's test on paired per-item predictions, for the last seed run
    mcnemar_result = None
    if HAS_STATSMODELS and last_seed_results is not None:
        preds_w2v = np.array(last_seed_results["word2vec"]["predictions"])
        preds_bert = np.array(last_seed_results["bert"]["predictions"])
        y_true = np.array(last_seed_test_labels)

        w2v_correct = (preds_w2v == y_true)
        bert_correct = (preds_bert == y_true)

        both_correct = np.sum(w2v_correct & bert_correct)
        only_w2v_correct = np.sum(w2v_correct & ~bert_correct)
        only_bert_correct = np.sum(~w2v_correct & bert_correct)
        both_wrong = np.sum(~w2v_correct & ~bert_correct)

        contingency_table = [[both_correct, only_w2v_correct],
                             [only_bert_correct, both_wrong]]
        mcnemar_result = mcnemar(contingency_table, exact=True)
        print(f"\nMcNemar's test on paired per-item predictions (seed={seeds[-1]}): "
              f"statistic={mcnemar_result.statistic:.4f}, p={mcnemar_result.pvalue:.4f}")
        print(f"Contingency table [[both_correct, only_w2v_correct], "
              f"[only_bert_correct, both_wrong]] = {contingency_table}")
        if mcnemar_result.pvalue < 0.05:
            print("-> The disagreement pattern between word2vec and BERT on individual "
                  "test items is statistically significant (p < 0.05).")
        else:
            print("-> The disagreement pattern is not statistically significant on this seed.")
    elif not HAS_STATSMODELS:
        print("\n[McNemar's test skipped] install statsmodels to enable it: pip install statsmodels")

    return per_seed_df, summary, {"t_stat": t_stat, "p_value": p_value}, mcnemar_result


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print("LOADING SHARED MODELS (word2vec + BERT)")
    print("=" * 70)
    w2v_model, _ = load_word2vec_model(pretrained=True)
    tokenizer, bert_model = load_bert_model()

    # ---------------- PART 1: MULTI-BENCHMARK WORD SIMILARITY ----------------
    print("\n" + "=" * 70)
    print("PART 1: MULTI-BENCHMARK WORD SIMILARITY "
          "(WordSim-353 + SimLex-999 + SimVerb-3500 + MEN + SCWS)")
    print("=" * 70)
    comparison_df, pair_dfs = run_multi_benchmark_word_similarity(
        w2v_model, tokenizer, bert_model,
        wordsim_path="/content/Datasets_W2VvsBERT/combined.csv",
        simlex_path="/content/Datasets_W2VvsBERT/SimLex-999.txt",
        simverb_path="/content/Datasets_W2VvsBERT/SimVerb-3500.txt",
        men_path="/content/Datasets_W2VvsBERT/MEN_dataset_natural_form_full.txt",
        scws_path="/content/Datasets_W2VvsBERT/ratings.txt"
    )
    comparison_df.to_csv("word_similarity_multi_benchmark_summary.csv", index=False)
    for name, pdf in pair_dfs.items():
        if pdf is not None:
            pdf.to_csv(f"word_similarity_{name}_pairs.csv", index=False)

    # ---------------- QUANTIFIED POLYSEMY EXPERIMENT ----------------
    print("\n" + "=" * 70)
    print("QUANTIFIED POLYSEMY EXPERIMENT")
    print("=" * 70)
    polysemy_df = run_polysemy_experiment(tokenizer, bert_model, w2v_model)
    polysemy_df.to_csv("polysemy_results.csv", index=False)
    interp = interpret_polysemy_results(polysemy_df)

    # ---------------- LAYER-WISE PROBING (word similarity) ----------------
    print("\n" + "=" * 70)
    print("LAYER-WISE PROBING: WORD SIMILARITY")
    print("=" * 70)
    df_wordsim = load_wordsim353("/content/Datasets_W2VvsBERT/combined.csv")
    layer_sim_df = layer_wise_word_similarity_probe(df_wordsim, w2v_model, tokenizer, bert_model)
    layer_sim_df.to_csv("layerwise_word_similarity.csv", index=False)

    # ---------------- PART 2: MULTI-SEED TEXT CLASSIFICATION ----------------
    print("\n" + "=" * 70)
    print("PART 2: MULTI-SEED SST-2 TEXT CLASSIFICATION (5 seeds)")
    print("=" * 70)
    per_seed_df, cls_summary, ttest_result, mcnemar_result = run_multi_seed_classification(
        w2v_model, tokenizer, bert_model, seeds=(10, 42, 123, 999, 2024)
    )
    per_seed_df.to_csv("classification_multi_seed_results.csv", index=False)

    # single-seed (seed=42) run kept for the layer-wise probe and error analysis,
    # which need one fixed dataset rather than an aggregate across seeds
    train_texts, train_labels, test_texts, test_labels = load_sst2_stratified_subsample(
        n_train_per_class=250, n_test_per_class=100, seed=42
    )
    class_results = run_classification_experiment(
        train_texts, train_labels, test_texts, test_labels, w2v_model, tokenizer, bert_model
    )
    disagreements_df = error_analysis(test_texts, test_labels, class_results)
    disagreements_df.to_csv("classification_disagreements.csv", index=False)

    # ---------------- LAYER-WISE PROBING (classification) ----------------
    print("\n" + "=" * 70)
    print("LAYER-WISE PROBING: TEXT CLASSIFICATION")
    print("=" * 70)
    layer_clf_df = layer_wise_classification_probe(
        train_texts, train_labels, test_texts, test_labels, tokenizer, bert_model
    )
    layer_clf_df.to_csv("layerwise_classification.csv", index=False)

    # ---------------- SUMMARY ----------------
    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    ws_row = comparison_df[comparison_df["dataset"] == "WordSim-353"].iloc[0]
    summary = pd.DataFrame({
        "metric": [
            "Benchmarks evaluated",
            "WordSim-353 corr. (w2v vs bert)",
            "WordSim-353 corr. diff (BERT-w2v), 95% CI",
            "Mean polysemy gap (within-across sense sim)",
            "Best BERT layer (word similarity)",
            "Best BERT layer (classification)",
            "Multi-seed accuracy (mean +/- std)",
            "Multi-seed F1 (mean +/- std)",
            "Paired t-test across seeds (BERT vs w2v F1)",
            "McNemar's test (last seed, per-item)",
            "Embedding extraction time ratio (BERT/word2vec)",
        ],
        "value": [
            ", ".join(comparison_df["dataset"].tolist()),
            f"w2v={ws_row['corr_w2v']:.4f}, bert={ws_row['corr_bert']:.4f}",
            f"{ws_row['corr_diff']:.4f} [{ws_row['ci_low']:.4f}, {ws_row['ci_high']:.4f}]",
            f"bert={polysemy_df['bert_gap'].mean():.4f}, word2vec=0.0000",
            f"layer {int(layer_sim_df.loc[layer_sim_df['spearman_corr_word_similarity'].idxmax(), 'layer'])}",
            f"layer {int(layer_clf_df.loc[layer_clf_df['f1'].idxmax(), 'layer'])}",
            f"w2v={cls_summary['w2v_acc_mean']:.4f}+/-{cls_summary['w2v_acc_std']:.4f}, "
            f"bert={cls_summary['bert_acc_mean']:.4f}+/-{cls_summary['bert_acc_std']:.4f}",
            f"w2v={cls_summary['w2v_f1_mean']:.4f}+/-{cls_summary['w2v_f1_std']:.4f}, "
            f"bert={cls_summary['bert_f1_mean']:.4f}+/-{cls_summary['bert_f1_std']:.4f}",
            f"t={ttest_result['t_stat']:.4f}, p={ttest_result['p_value']:.4f}",
            (f"stat={mcnemar_result.statistic:.4f}, p={mcnemar_result.pvalue:.4f}"
             if mcnemar_result is not None else "skipped (statsmodels not installed)"),
            f"{class_results['bert']['embedding_extraction_time'] / max(class_results['word2vec']['embedding_extraction_time'], 1e-9):.1f}x",
        ],
    })
    print(summary.to_string(index=False))
    summary.to_csv("summary_results.csv", index=False)
    print("\nAll results saved to CSV files in the working directory.")


if __name__ == "__main__":
    main()

LOADING SHARED MODELS (word2vec + BERT)
Loading pretrained word2vec model (this may take a while)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



PART 1: MULTI-BENCHMARK WORD SIMILARITY (WordSim-353 + SimLex-999 + SimVerb-3500 + MEN + SCWS)

[WordSim-353] word2vec OOV drop rate: 3/353 pairs (0.85%) excluded due to at least one out-of-vocabulary word.
[WordSim-353] Spearman (word2vec vs human): 0.6941 (n=350)
[WordSim-353] Spearman (BERT vs human): 0.6704 (n=350)
[WordSim-353] Bootstrap 95% CI (BERT - word2vec): -0.0237 [-0.0970, 0.0476]
[WordSim-353] -> Difference is NOT statistically significant (CI includes 0).

[SimLex-999] word2vec OOV drop rate: 0/999 pairs (0.00%) excluded due to at least one out-of-vocabulary word.
[SimLex-999] Spearman (word2vec vs human): 0.4420 (n=999)
[SimLex-999] Spearman (BERT vs human): 0.4504 (n=999)
[SimLex-999] Bootstrap 95% CI (BERT - word2vec): 0.0085 [-0.0398, 0.0551]
[SimLex-999] -> Difference is NOT statistically significant (CI includes 0).

[SimVerb-3500] word2vec OOV drop rate: 0/3500 pairs (0.00%) excluded due to at least one out-of-vocabulary word.
[SimVerb-3500] Spearman (word2vec vs

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

...

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

[seed=10] Train size: 500, Test size: 200
[seed=10] w2v acc=0.8050 f1=0.8044 | bert acc=0.7750 f1=0.7747
Loading SST-2 stratified subsample (seed=42)...
[seed=42] Train size: 500, Test size: 200
[seed=42] w2v acc=0.7600 f1=0.7596 | bert acc=0.8000 f1=0.7999
Loading SST-2 stratified subsample (seed=123)...
[seed=123] Train size: 500, Test size: 200
[seed=123] w2v acc=0.7900 f1=0.7899 | bert acc=0.8350 f1=0.8349
Loading SST-2 stratified subsample (seed=999)...
[seed=999] Train size: 500, Test size: 200
[seed=999] w2v acc=0.8350 f1=0.8350 | bert acc=0.8600 f1=0.8599
Loading SST-2 stratified subsample (seed=2024)...
[seed=2024] Train size: 500, Test size: 200
[seed=2024] w2v acc=0.7900 f1=0.7895 | bert acc=0.8150 f1=0.8137

=== MULTI-SEED SST-2 CLASSIFICATION SUMMARY ===
 seed  w2v_accuracy   w2v_f1  bert_accuracy  bert_f1
   10         0.805 0.804408          0.775 0.774724
   42         0.760 0.759615          0.800 0.799920
  123         0.790 0.789916          0.835 0.834897
  999     